In [1]:

%load_ext autoreload
%autoreload 2

In [9]:
import os
import sys

ruta_proyecto = os.path.abspath("..")
if ruta_proyecto not in sys.path:
    sys.path.append(ruta_proyecto)

In [11]:

from torch.utils.data import DataLoader
from pathlib import Path
from utils.get_woof import ImagewoofColorizationDataset

import torch
import sys
from utils.trainer import trainer
DATA_DIR = Path("../imagewoof2-160")
BATCH_SIZE = 8
EPOCHS = 5

train_ds = ImagewoofColorizationDataset(DATA_DIR, split="train")
val_ds   = ImagewoofColorizationDataset(DATA_DIR, split="val")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=0)

In [ ]:
def train_model(model, train_loader, val_loader, save_name, criterion="l1"):
    save_path = "pesos_entrenados"
    model_path = Path(save_path) / save_name

    # carpeta donde guardamos las curvas loss vs epoch
    history_dir = Path("loss_vs_epoch")
    history_dir.mkdir(parents=True, exist_ok=True)
    history_path = history_dir / f"{save_name}_history.pt"

    train = False  # Cambia a True para forzar el reentrenamiento

    # Verificar si ya existe un modelo entrenado
    if model_path.exists() and not train:
        print(f"✅Modelo ya entrenado encontrado en '{model_path}'.")
        print("No se vuelve a entrenar para evitar sobreescritura.")

        # (opcional) si ya tenés la history guardada, podés devolverla:
        if history_path.exists():
            print(f"History encontrada en '{history_path}'.")
            history = torch.load(history_path, map_location="cpu")
            return history
        else:
            print("⚠️No se encontró history guardada para este modelo.")
            return None

    else:
        print("🚀No se encontró modelo entrenado, iniciando entrenamiento...")

        # ⬇️ahora trainer devuelve la history
        history = trainer(
            model,
            train_loader,
            val_loader,
            epochs=10,
            save_path=save_path,
            save_name=save_name,
            criterion=criterion,
        )

        print(f"💾Modelo guardado en: {model_path}")

        # ⬇️guardamos la history de forma GENERAL
        torch.save(history, history_path)
        print(f"📈History guardada en: {history_path}")

        return history

## Modelo con bacbone resnet, entrenado con L1 + Histograma

In [16]:
from models.unet_resnet34 import get_model_unet_resnet34
from utils.trainer import train_model

In [17]:
import os

model = get_model_unet_resnet34(pre_entrenado=True, congelar_encoder=False)
save_name = "unet_resnet34_histogram.pt"
train_model(model, train_loader, val_loader, save_name, criterion="histogram")

/Users/damiandistefano/Documents/UDESA/3_año/2_semestre/Vision artificial/TPs/Tp_final_vision/utils/trainer.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  


🆕 Modelo no encontrado. Entrenando 'unet_resnet34_histogram.pt'...
🚀 Iniciando entrenamiento en: cpu
📉 Criterio: histogram


Ep 1/10 [Train]:   0%|          | 0/1129 [00:00<?, ?it/s]/Users/damiandistefano/Documents/UDESA/3_año/2_semestre/Vision artificial/TPs/Tp_final_vision/utils/trainer.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  
Ep 1/10 [Train]:   1%|          | 7/1129 [00:11<31:53,  1.71s/it, loss=0.2569]


KeyboardInterrupt: 